# 4 구현

## 4-1 아이디어를 코드로 바꾸는 구현

### 상하좌우

In [ ]:
n = int(input())
iti = list(input().split())
dir = {
    "U": [0, 1],
    "D": [0, -1],
    "L": [-1, 0],
    "R": [1, 0]
}
loc = [1, 1]

for p in iti:
    nx = loc[0] + dir[p][0]
    ny = loc[1] + dir[p][1]
    if 1 <= nx <= n and 1 <= ny <= n:
        loc[0] = nx
        loc[1] = ny

print(loc[0], loc[1])

- 튜플: 변경 불가 자료형, 리스트보다 메모리 사용 적고 빠름
- 리스트에 튜플 더하기 불가
- 좌표 범위 확인 시 원소 하나하나 확인해줘야(nx, ny 정의 필요)
- 조건문에서 1 <= nx <= n 사용 가능
- 나는 현재좌표와 UDLR을 딕셔너리로 정의
- 모범답안은 dx,dx,move_types를 정의하고 인덱스 맞춰 사용, continue문을 이동 수행 전에 사용
- 내 코드가 의미기반으로 더 직관적이지만 모범답안이 조금 더 빠름(둘다 복잡도 O(1)), 표준 코테 스타일, 확장성 좋음
-  key lookup + value access(hash table에 접근)- 딕셔너리 연산 lookup 시간과 방향 많아지면 코드 더 길어질 수 있음. dir[p]에서 딕셔너리 탐색, 문자열 key 처리: "U" → dict lookup → (0,1)
- index lookup + array access(dx/dy list indexing 방식, 배열 2개만 접근)- 확장성이 좋고 빠름, 문자열 → index로 1번만 변환하면 끝,: "U" → dict lookup → (0,1)
- 아래는 모범답안의 최적화 버전: move_types의 인덱스를 짝맞춰 for문 돌리던 부분을 딕셔너리로 인덱스 짝을 맞춰줌

In [ ]:
#모범답안 개선할 부분
for i in range(len(move_types)):
    if plan == move_types[i]:
        nx = x + dx[i]
        ny = y + dy[i]

#최적화
move_types = {'L':0,'R':1,'U':2,'D':3}
nx = x + dx[move_types[plan]]
ny = y + dy[move_types[plan]]

### 시각

In [ ]:
n = int(input())
count = 0
for i in range(n + 1):
    for j in range(60):
        for k in range(60):
            if '3' in str(i) + str(j) + str(k):
                count += 1
print(count)

- in 연산자: 리스트에서 요소 검색, 문자열에서 문자열 검색
- n + 1

## 4-2 왕실의 나이트

In [ ]:
p = input()
col = {'a':1,'b':2,'c':3,'d':4,'e':5,'f':6,'g':7,'h':8}
c , r = col[p[0]], int(p[1])
count = 0
if (c + 2 <= 8 and r + 1 <= 8): count += 1
if (c + 2 <= 8 and r - 1 >= 1): count += 1
if (c - 2 >= 1 and r + 1 <= 8): count += 1
if (c - 2 >= 1 and r - 1 >= 1): count += 1
if (c + 1 <= 8 and r + 2 <= 8): count += 1
if (c + 1 <= 8 and r - 2 >= 1): count += 1
if (c - 1 >= 1 and r + 2 <= 8): count += 1
if (c - 1 >= 1 and r - 2 >= 1): count += 1
print(count)

- 상하좌우처럼 경우의수 한개한개 값 비교해서 카운트
- 패턴이 반복되는 하드코딩인걸 알지만 함수화 실패
- 모범답안은 딕셔너리가 아닌 `int(ord('a'))`로 계산, 8가지 방향을 정의
- 이동 가능 if에 사방에 대한 조건을 and로 넣어 카운트

# 4-3 게임 개발

In [ ]:
n, m = map(int, input().split())
x, y, d = map(int, input().split())

graph = [list(map(int, input().split())) for _ in range(n)]
visited = [[False for _ in range(m)] for _ in range(n)]
visited[x][y] = True

dx = [0, -1, 0, 1]
dy = [-1, 0, 1, 0]
count = 0

while True:
    nx, ny = x + dx[d], y + dy[d]
    if 0 <= nx < m and 0 <= ny < n and graph[ny][nx] == 0 and visited[nx][ny] == False:
        x, y = nx, ny
        visited[x][y] = True
        d -= 1
        if d == 0: d = 3
        count += 1
    elif 0 <= nx < m and 0 <= ny < n and graph[ny][nx] == 0 and visited[nx][ny] == True:
        d += 1
    elif 0 <= (x - dx[d]) < m and 0 <= (y - dy[d]) < n and all(visited[x+dx[i]][y+dy[i]] == True or graph[x+dx[i]][y+dy[i]] == 1 for i in range(4)):
        x, y = x - dx[d], y - dy[d]
    else:
        break
    
print(count)

- 배열 인덱스가 `[x][y]`인지 `[y][x]` 잘 생각
- ` 0 <= nx < m`은 체크할 필요 없이 `graph`와 `visited`의 값만으로도 갈수있는지 확인됨
- 변수명 `map`은 함수명과 같아서 주의
- 시작점 방문처리
- `if d == -1: d = 3` 대신 `d = (d - 1) % 4`도 가능하며 모범답안은 `turn_left()`를 정의, 그리고 왼쪽 회전 -1??: 강원도가 동해잖아.. 정동진...
- "네 방향 모두 이미 가본 칸이거나 바다로 되어있는 칸"의 조건 확인은 `if all(not(갈수있음) for i in range(4)):`로 작성 가능하지만 발상 자체가 비효율적이고 작동도 안됨
- 모범답안처럼 `turn_time == 4`이면 네 방향 모두 못간다는 것을 알아야 함(turn counter로 상태 관리)